In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rc
import scipy.integrate

import optimal_consumption as oc

In [ ]:
rc("font", family="serif", serif=["Computer Modern"], size=18)
rc("text", usetex=True)

In [ ]:
R = 2


def delta(y):
    return 0.02


def r(y):
    return 0.013


lmbda_val = 1.66


def lmbda(y):
    return lmbda_val * np.sqrt(y)


def sigma(y):
    return np.sqrt(y)


rho = -0.84

kappa = 0.088


def a(y):
    return -kappa * (y - 0.035)


nu = 0.031


def b(y):
    return nu * np.sqrt(y)

In [ ]:
eta = oc.myopic_consumption_rate(R, r, delta, lmbda)

In [ ]:
X = np.linspace(0, 1, 1000)

for n in [2, 10, 25, 100]:
    X_sol, sol = oc.optimal_consumption_rate(
        R, r, delta, lmbda, rho, a, b, 1 / n, np.sqrt(n), 10000
    )
    plt.plot(X, 100 * np.interp(X, X_sol, sol, left=np.nan), label=f"m={n}")

plt.plot(X, 100 * eta(X), color="grey", linestyle="dashed", label="$\\eta(y)$")

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Consumption Rate (\\%)")

plt.savefig("img/heston/convergence_neumann.pdf", bbox_inches="tight")

In [ ]:
X_sol, sol = oc.optimal_consumption_rate(
    R, r, delta, lmbda, rho, a, b, 1 / 10000, np.sqrt(10000), 10000
)

In [ ]:
X = np.linspace(0.01, 7.5, 100)
sol_pf = oc.optimal_investment_fraction(R, sigma, lmbda, rho, b, X_sol, sol)

plt.plot(X, 100 * np.interp(X, X_sol, sol_pf), label="Optimal")
plt.plot(X, 100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X), label="Myopic")
plt.fill_between(
    X,
    100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X),
    100 * np.interp(X, X_sol, sol_pf),
    color="green",
    alpha=0.3,
    hatch="//",
    label="Intertemporal Hedge",
)

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Risky Asset Allocation (\\%)")

plt.savefig("img/heston/portfolio_limit.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(0.01, 7.5, 1000)

plt.plot(X, 100 * (np.interp(X, X_sol, sol) - eta(X)), label="$\\xi_h$")
plt.axhline(
    100 * (-(kappa + (R - 1) / R * rho * lmbda_val * nu)),
    color="grey",
    linestyle="dashed",
    label="$-\\tilde{\\kappa}$",
)

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Consumption Adjustment (\%)")

plt.savefig("img/heston/consumption_limit.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(0.01, 0.06, 1000)

plt.plot(X, 100 * np.interp(X, X_sol, sol), label="Optimal")
plt.plot(X, 100 * eta(X), label="Myopic")
plt.legend()
plt.xlabel("$y$")
plt.ylabel("Consumption Rate (\\%)")
plt.savefig("img/heston/consumption.pdf", bbox_inches="tight")
plt.show()

plt.plot(X, 100 * np.interp(X, X_sol, sol_pf), label="Optimal")
plt.plot(X, 100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X), label="Myopic")
plt.fill_between(
    X,
    100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X),
    100 * np.interp(X, X_sol, sol_pf),
    color="green",
    alpha=0.3,
    hatch="//",
    label="Intertemporal Hedge",
)
plt.ylabel("Risky Asset Allocation (\\%)")
plt.legend()
plt.xlabel("$y$")
plt.savefig("img/heston/portfolio.pdf", bbox_inches="tight")
plt.show()

# Approximate Solutions from Guasoni & Wang

In [ ]:
b_gw = 0.088
rho = -0.84
mu = 1.66
r_gw = 0.013
beta = 0.02
gamma = 2
sigma_gw = 1
theta = 0.035
a_gw = 0.031
Sigma = sigma_gw**2
A = a_gw**2

In [ ]:
k = b_gw - ((1 - gamma) * rho * a_gw * mu) / (gamma * sigma_gw)
K = (beta + (gamma - 1) * r_gw) / gamma

c = ((gamma - 1) * mu**2) / (2 * gamma**2 * Sigma)
alpha = np.sqrt(k**2 + 2 * c * A)

In [ ]:
def B(t):
    return (
        2 * c * (np.exp(alpha * t) - 1) / (np.exp(alpha * t) * (k + alpha) - k + alpha)
    )


def C(t):
    return (
        -(2 * b_gw * theta)
        / A
        * (
            np.log((k + alpha) * np.exp(alpha * t) - k + alpha)
            - np.log(2 * alpha)
            - 0.5 * (k + alpha) * t
        )
        - K * t
    )


def upper_bound(y):
    return 1 / scipy.integrate.quad(lambda t: np.exp(C(t) - B(t) * y), 0, 1000)[0]

In [ ]:
def eq(c, alpha, Q):
    return [
        c
        - (
            ((gamma - 1) * mu**2) / (2 * gamma**2 * Sigma)
            - (gamma - 1) * A * (1 - rho**2) * Q
        ),
        alpha - np.sqrt(k**2 + 2 * c * A),
        Q - (2 * c**2) / (k + alpha) ** 2,
    ]


x = scipy.optimize.root(lambda x: eq(x[0], x[1], x[2]), [1, 1, 1]).x
c_bar = x[0]
alpha_bar = x[1]
Q_bar = x[2]


def B_bar(t):
    return (
        2
        * c_bar
        * (np.exp(alpha_bar * t) - 1)
        / (np.exp(alpha_bar * t) * (k + alpha_bar) - k + alpha_bar)
    )


def C_bar(t):
    return (
        -(2 * b_gw * theta)
        / A
        * (
            np.log((k + alpha_bar) * np.exp(alpha_bar * t) - k + alpha_bar)
            - np.log(2 * alpha_bar)
            - 0.5 * (k + alpha_bar) * t
        )
        - K * t
    )


def lower_bound(y):
    return (
        1 / scipy.integrate.quad(lambda t: np.exp(C_bar(t) - B_bar(t) * y), 0, 1000)[0]
    )

In [ ]:
plt.plot(
    X := np.linspace(0.01, 0.06, 100),
    np.vectorize(upper_bound)(X) * 100,
    color="gray",
    linestyle="dashed",
    label="A priori bounds",
)
plt.plot(X, np.interp(X, X_sol, sol) * 100, label="ODE Solution")
plt.plot(X, np.vectorize(lower_bound)(X) * 100, color="grey", linestyle="dashed")

plt.legend()
plt.xlabel("$y$")
plt.ylabel("Consumption Rate (\\%)")

plt.savefig("img/heston/apriori.pdf", bbox_inches="tight")